# Phase 8 — Sequential annealing via freeze / unfreeze

Demonstrates the *worst-actor* annealing recipe for pushing $|F|_{\max}$ below the joint-optimization plateau at $\phi=0.86$.

**The recipe per round:**
1. Identify the worst-balanced particle in the current state via `s.eval_force_balance()`
2. **Freeze** all other particles' DOFs (by masking the force/gradient on them)
3. Optimize the lone particle (FIRE + L-BFGS) — drives its solo residual to machine precision
4. **Unfreeze** everyone, joint-optimize the full system to its new plateau
5. Track best-of-all-rounds; only accept new state if global $|F|_{\max}$ improved

**Why this works (sometimes):** the joint optimization plateau at $|F|_{\max}\!\sim\!2.5\!\times\!10^{-3}$ is set by *multi-particle frustration*, not single-particle physics. Anneal-then-rejoin can steer the joint optimizer into a deeper basin — but only in basin-compatible cases. Best-of tracking guards against the non-monotonic regressions observed in practice.

**Pre-requisite state file:** the notebook loads `../../tmp/phase8_stability/post_swell_state.npz`. If you don't have it, run `python_v2/notebooks/phase8_adam_quench.ipynb` first to generate a φ=0.86 starting state, or adjust `INPUT` below.

## 1. Setup

In [ ]:
import sys, os, time
sys.path.insert(0, '..')                                          # python_v2 root
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import numpy as np
import tensorflow as tf
import src.simulation.tf_sim as tfm; tfm.set_dtype(tf.float64)
from src.simulation.tf_sim import (internal_forces_tf, k_reg_forces_tf,
                                     inter_capsule_forces_tf, primitive_forces_tf)
from src.epd.particles import ParticleSpec
from src.epd.system import System
from scipy.optimize import minimize
import matplotlib.pyplot as plt

P, N  = 16, 60
SHAPE = (P, N, 2)
INPUT = '../../tmp/phase8_stability/post_swell_state.npz'   # post-swell at φ=0.86

# Round budgets — demo-sized. For production, push joint counts to 25k FIRE + 10k LBFGS.
SINGLE_FIRE   = 3_000      # single-particle FIRE
SINGLE_LBFGS  = 1_000      # single-particle L-BFGS
JOINT_FIRE    = 8_000      # joint FIRE
JOINT_LBFGS   = 3_000      # joint L-BFGS
N_ROUNDS      = 4

In [ ]:
# Build system, load post-swell state, zero all damping (we're doing pure quench)
def build_load():
    s = System(18.0, 18.0, periodic_x=True, periodic_y=True,
               dt_factor=0.25, candidacy_kind='prcm', E_candidates=9)
    s.add_particles(ParticleSpec(count=P, type='emulsion', gamma=1.0, kappa=0.02,
                                  N_nodes=N, Oh=5.0,
                                  poly_dist={'type':'bimodal','ratio':0.5,'delta':0.2}))
    s.initialize(phi_target=0.49, seed=42, verbose=False, relax_only=True, n_relax_init=0)
    s.restore_state(INPUT)
    DTYPE = s._params['_alpha_tf'].dtype
    s._params['xi_drag_per_p']    = tf.zeros([P], dtype=DTYPE)
    s._params['_alpha_tf']        = tf.constant(0.0, dtype=DTYPE)
    s._params['alpha_damp_per_p'] = tf.zeros([P], dtype=DTYPE)
    s.beta_rb = 0.0
    return s

s = build_load()
print(f'φ = {s.phi_outer:.4f}   P = {P}   N_nodes = {N}   box = {s._params["box"].numpy()}')

## 2. Masked-force helpers

**Freeze pattern:** there is no `System.freeze_particles(idx)` method. Instead we multiply forces and gradients by a 3D mask of shape `(P, N, 2)` where `mask[p, n, :] = 1` if particle `p`'s node `n` is **free**, `0` if **frozen**. FIRE and L-BFGS see zero force on frozen DOFs, so they don't move. This is the **mask-as-freeze** idiom.

- `mask_full`: ones everywhere → joint optimization (all particles free)
- `mask_single[p] = 1; rest = 0` → only particle `p` is free (single-particle solve)
- For multi-particle subsets (e.g. free 3 worst, freeze the rest), set the corresponding rows of `mask` to 1.

In [ ]:
def force_diff(x_all, caps, params, prim_data):
    """Pure-tensor force kernel (gradient-flowing for L-BFGS GradientTape).
    Pass g=0.0 as a Python float to internal_forces_tf to avoid tf.cond+XLA gradient bug."""
    N_nodes = x_all.shape[1]
    r_c_flat = tf.repeat(params['r_c_per_p'], N_nodes)
    k_c_flat = tf.repeat(params['k_c_per_p'], N_nodes)
    L0_flat  = tf.repeat(params['L0'],        N_nodes)
    box      = params.get('box', None)
    f_c = inter_capsule_forces_tf(x_all, caps, r_c_flat, k_c_flat, L0_flat, box=box)
    if prim_data is not None:
        f_c = f_c + primitive_forces_tf(x_all, tf.constant(0.0, dtype=x_all.dtype),
                                         prim_data, r_c_flat, k_c_flat, L0_flat, box=box)
    return (internal_forces_tf(x_all, params, 0.0)
            + k_reg_forces_tf(x_all, params) + f_c)


def measure_Fmax(s, mask3d=None):
    """max |F| (over masked DOFs if mask given, else all)."""
    caps = tf.constant(s._cm_mgr.CapCandidates, dtype=tf.int32)
    F = force_diff(s._state['x_all'], caps, s._params, s._prim_data).numpy()
    if mask3d is not None:
        F = F * mask3d
    return float(np.linalg.norm(F, axis=2).max())


def run_fire_masked(s, mask3d, n_steps, theta_np,
                     dt_init=0.005, dt_max=0.03, alpha_start=0.05, N_min=20,
                     prcm_every=100):
    """FIRE quench with a freeze-mask: frozen DOFs see zero force and zero velocity."""
    DTYPE = s._state['x_all'].dtype
    x_var = tf.Variable(s._state['x_all'], dtype=DTYPE)
    v_var = tf.Variable(tf.zeros_like(x_var), dtype=DTYPE)
    mask  = tf.constant(mask3d, dtype=DTYPE)
    s._cm_mgr.update(s._state['x_cm'].numpy(), theta_np, x_all=s._state['x_all'].numpy())
    dt = dt_init; alpha = alpha_start; n_pos = 0
    F_INC, F_DEC, F_ALPHA = 1.1, 0.5, 0.99
    TINY = tf.constant(1e-30, dtype=DTYPE)
    for i in range(n_steps):
        caps = tf.constant(s._cm_mgr.CapCandidates, dtype=tf.int32)
        f = force_diff(x_var, caps, s._params, s._prim_data) * mask   # ← freeze applied here
        Pn = float(tf.reduce_sum(f * v_var).numpy())
        v_norm = tf.sqrt(tf.reduce_sum(v_var * v_var) + TINY)
        f_norm = tf.sqrt(tf.reduce_sum(f * f) + TINY)
        v_var.assign((1.0 - alpha) * v_var + (alpha * v_norm / f_norm) * f)
        if Pn > 0.0 and n_pos > N_min:
            dt = min(dt * F_INC, dt_max); alpha = alpha * F_ALPHA
        if Pn <= 0.0:
            dt = dt * F_DEC; v_var.assign(tf.zeros_like(v_var))
            alpha = alpha_start; n_pos = 0
        else:
            n_pos += 1
        v_var.assign_add(dt * f)
        x_var.assign_add(dt * v_var * mask)                            # ← freeze applied here
        if (i + 1) % prcm_every == 0:
            x_np = x_var.numpy()
            if s._cm_mgr.needs_update(x_np.mean(axis=1), theta_np):
                s._cm_mgr.update(x_np.mean(axis=1), theta_np, x_all=x_np)
    s._state['x_all'] = x_var.value()
    s._state['x_cm']  = tf.reduce_mean(x_var, axis=1)


def run_lbfgs_masked(s, mask3d, max_iter, theta_np, prcm_every=50):
    """L-BFGS-B on 0.5*||F·mask||² — gradient through GradientTape, mask applied to f AND grad."""
    DTYPE = s._state['x_all'].dtype
    mask = tf.constant(mask3d, dtype=DTYPE)
    x_init = s._state['x_all'].numpy().reshape(-1)
    x_var = tf.Variable(s._state['x_all'], dtype=DTYPE)
    n_evals = [0]
    def f_and_grad(x_flat):
        n_evals[0] += 1
        x_3d = tf.constant(x_flat.reshape(SHAPE), dtype=DTYPE)
        x_var.assign(x_3d)
        if n_evals[0] % prcm_every == 0:
            x_np = x_var.numpy()
            if s._cm_mgr.needs_update(x_np.mean(axis=1), theta_np):
                s._cm_mgr.update(x_np.mean(axis=1), theta_np, x_all=x_np)
        caps = tf.constant(s._cm_mgr.CapCandidates, dtype=tf.int32)
        x_local = tf.Variable(x_3d, trainable=True)
        with tf.GradientTape() as tape:
            F = force_diff(x_local, caps, s._params, s._prim_data) * mask
            L = 0.5 * tf.reduce_sum(F * F)
        g = tape.gradient(L, x_local).numpy() * mask3d                  # ← freeze applied here
        return float(L.numpy()), g.reshape(-1)
    res = minimize(f_and_grad, x_init, method='L-BFGS-B', jac=True,
                   options={'maxiter': max_iter, 'maxfun': max_iter*10,
                            'gtol': 1e-20, 'ftol': 1e-20, 'disp': False})
    s._state['x_all'] = tf.constant(res.x.reshape(SHAPE), dtype=DTYPE)
    s._state['x_cm']  = tf.reduce_mean(s._state['x_all'], axis=1)
    return res.nit


def save_snapshot(s):
    return {'x_all': s._state['x_all'].numpy().copy(),
            'x_cm':  s._state['x_cm'].numpy().copy()}

def restore_snapshot(s, snap):
    DTYPE = s._state['x_all'].dtype
    s._state['x_all'] = tf.constant(snap['x_all'], dtype=DTYPE)
    s._state['x_cm']  = tf.constant(snap['x_cm'],  dtype=DTYPE)

print('helpers defined.')

## 3. Baseline joint optimization (no freezing)

Establish the plateau for the **all-particles-free** joint solve. This is the floor we're trying to beat.

In [ ]:
theta_np  = s._state['theta'].numpy()
mask_full = np.ones(SHAPE)

F_initial = measure_Fmax(s)
print(f'Initial |F|_max (before any opt): {F_initial:.3e}\n')

print(f'Baseline joint opt: FIRE {JOINT_FIRE} + LBFGS {JOINT_LBFGS}...')
t0 = time.time()
run_fire_masked(s, mask_full, JOINT_FIRE, theta_np)
F_after_fire = measure_Fmax(s)
nit = run_lbfgs_masked(s, mask_full, JOINT_LBFGS, theta_np)
F_baseline = measure_Fmax(s)
print(f'  FIRE  →  |F|_max = {F_after_fire:.3e}')
print(f'  LBFGS →  |F|_max = {F_baseline:.3e}  ({nit} iters)  [wall {time.time()-t0:.1f}s]')

baseline_snap = save_snapshot(s)
trajectory = [('baseline', F_baseline, -1, None)]

## 4. One annealing round — the freeze / unfreeze pattern in detail

This cell does ONE round with verbose printing so you can see each step.

In [ ]:
# ── A) Identify the worst actor ────────────────────────────────────────────
fb = s.eval_force_balance()
worst_p = int(fb['top_stuck'][0][0])
worst_F = fb['top_stuck'][0][2]
print(f'A)  worst actor:  p = {worst_p}   |F| = {worst_F:.3e}')
print(f'    top 3 stuck:  {[(int(p), f"{f:.2e}") for p, _, f in fb["top_stuck"][:3]]}')

# ── B) Freeze everyone else, optimize the worst particle to machine precision ──
mask_single = np.zeros(SHAPE)
mask_single[worst_p] = 1.0                       # only this particle is free
print(f'\nB)  freeze all except p={worst_p}, single-particle optimize...')
t0 = time.time()
run_fire_masked  (s, mask_single, SINGLE_FIRE,  theta_np)
run_lbfgs_masked(s, mask_single, SINGLE_LBFGS, theta_np)
F_solo  = measure_Fmax(s, mask_single)           # |F| of the free particle only
F_global_after_B = measure_Fmax(s)               # |F| of any particle (incl. frozen residuals)
print(f'    p={worst_p} solo |F|_max = {F_solo:.3e}   '
      f'(global |F|_max = {F_global_after_B:.3e})   [wall {time.time()-t0:.1f}s]')

# ── C) Unfreeze everyone, joint optimize ─────────────────────────────────────
print(f'\nC)  unfreeze all, joint optimize...')
t0 = time.time()
run_fire_masked  (s, mask_full, JOINT_FIRE,  theta_np)
run_lbfgs_masked(s, mask_full, JOINT_LBFGS, theta_np)
F_round1 = measure_Fmax(s)
print(f'    joint plateau |F|_max = {F_round1:.3e}   [wall {time.time()-t0:.1f}s]')

trajectory.append(('round1', F_round1, worst_p, F_solo))
print(f'\n→  baseline {F_baseline:.3e}  →  round 1 {F_round1:.3e}  '
      f'({"IMPROVED" if F_round1 < F_baseline else "REGRESSED"})')

## 5. Multi-round loop with **best-of tracking**

Rounds 2-N. Key safeguard: after each round, only **accept** the new state if global $|F|_{\max}$ improved. If a round regressed (which empirically happens after 3-4 rounds), **restore the best-so-far state** and continue searching from there.

This is the variant of the experiment that addresses the V-shape regression we observed: it locks in gains and prevents the system from drifting into a worse basin.

In [ ]:
best_snap = save_snapshot(s) if F_round1 < F_baseline else baseline_snap
best_F    = min(F_round1, F_baseline)
if F_round1 >= F_baseline:
    restore_snapshot(s, baseline_snap)
    print(f'[round 1 regressed — restored baseline snapshot]')

for rnd in range(2, N_ROUNDS + 1):
    print(f'\n{"="*60}\nRound {rnd}    (current best: {best_F:.3e})\n{"="*60}')
    fb = s.eval_force_balance()
    worst_p = int(fb['top_stuck'][0][0])
    print(f'  worst actor: p={worst_p}  |F|={fb["top_stuck"][0][2]:.3e}')

    mask_single = np.zeros(SHAPE); mask_single[worst_p] = 1.0
    t0 = time.time()
    run_fire_masked  (s, mask_single, SINGLE_FIRE,  theta_np)
    run_lbfgs_masked(s, mask_single, SINGLE_LBFGS, theta_np)
    F_solo = measure_Fmax(s, mask_single)
    print(f'  solo |F|_max = {F_solo:.3e}   [B wall {time.time()-t0:.1f}s]')

    t0 = time.time()
    run_fire_masked  (s, mask_full, JOINT_FIRE,  theta_np)
    run_lbfgs_masked(s, mask_full, JOINT_LBFGS, theta_np)
    F_new = measure_Fmax(s)
    print(f'  joint |F|_max = {F_new:.3e}   [C wall {time.time()-t0:.1f}s]')

    if F_new < best_F:
        best_F    = F_new
        best_snap = save_snapshot(s)
        verdict   = f'ACCEPT (new best: {best_F:.3e})'
    else:
        restore_snapshot(s, best_snap)
        verdict   = f'REJECT (restored best: {best_F:.3e})'
    print(f'  → {verdict}')
    trajectory.append((f'round{rnd}', F_new, worst_p, F_solo))

print(f'\n{"="*60}')
print(f'Done.   baseline {F_baseline:.3e}   →   best {best_F:.3e}   ({F_baseline/best_F:.1f}× drop)')

## 6. Summary + plot

In [ ]:
print(f'{"phase":<10}  {"|F|_max":>11}  {"worst_p":>8}  {"solo |F|":>11}')
print('-' * 46)
for name, fmax, p, p_solo in trajectory:
    p_str = str(p) if p >= 0 else '-'
    p_solo_str = f'{p_solo:.3e}' if p_solo is not None else '-'
    print(f'{name:<10}  {fmax:>11.3e}  {p_str:>8}  {p_solo_str:>11}')

labels = [t[0] for t in trajectory]
fmaxes = np.array([t[1] for t in trajectory])
running_best = np.minimum.accumulate(fmaxes)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(range(len(labels)), fmaxes,       'b-o', lw=1.3, label='round outcome')
ax.semilogy(range(len(labels)), running_best, 'r--', lw=1.5, label='best-of (running min)')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30)
ax.set_ylabel('global |F|_max  (after joint opt)')
ax.grid(alpha=0.3, which='both'); ax.legend()
ax.set_title(f'Sequential annealing — {F_baseline/best_F:.1f}× drop from baseline')
plt.tight_layout(); plt.show()

## 7. Visualize the best-of state

In [ ]:
restore_snapshot(s, best_snap)
fig, ax = plt.subplots(figsize=(7, 7))
s.set_color_palette('palette1', seed=1)
s.render(ax=ax)
ax.set_title(f'Best-of state   |F|_max = {best_F:.3e}   φ = {s.phi_outer:.4f}')
plt.show()

## Notes

**Adapting the freeze pattern to other use-cases:**

- *Free the worst K particles together*: set `mask3d[worst_indices] = 1`, rest = 0. Multi-particle solo solves may help when worst actors are coupled.
- *Free a contiguous cluster*: e.g. percolating chain of high-|F| contacts. Useful for shear-band ICs where the band geometry is known.
- *Per-node freezing*: `mask3d[p, n_subset, :] = 1` freezes only specific nodes within a particle.
- *Translation-mode removal*: subtract per-particle mean force before applying mask, to avoid spurious CM drift on the free particle.

**Production budgets:**

| Stage | Demo (this notebook) | Production |
| --- | --- | --- |
| Single FIRE / LBFGS | 3k / 1k | 5k / 2k |
| Joint FIRE / LBFGS | 8k / 3k | 25k / 10k |
| N_ROUNDS | 4 | 5-8 (with best-of) |

**Empirical observations from the phase-8 anneal study** (at φ=0.86, P=16):
- Single-particle solo always reaches machine precision (1e-11 to 1e-13) regardless of which particle.
- Joint plateau improves for 2-3 rounds (up to ~3× from baseline), then can regress.
- Best-of tracking locks in the gain — without it, you can end up worse than baseline.
- The improvement is real (different basin of the marginal-jamming landscape), not numerical.

**See also:** `papers/summary_of_methods/main.tex` — the perimeter-regularization $k_{\rm reg}$ pseudo-force discretization section discusses why the joint plateau exists in the first place.